In [1]:
import pandas as pd
from datetime import datetime, timedelta
from bs4 import BeautifulSoup as bs
import requests
from tqdm import tqdm
from time import sleep
import json

In [2]:
def get_top_elo(date=None):
    top_teams = pd.DataFrame()

    fir = ['team','elo_(k=32)','elo_(k=64)','glicko_1','glicko_2']
    sec = ['', 'avg_7d', 'δ7d', 'avg_30d', 'δ30d', 'rating', 'μ', 'σ', 'δrat.7d']
    
    url = f"https://www.datdota.com/ratings"
    page = requests.get(url)

    soup = bs(page.text,'html.parser')
    table_body = soup.find('table')

    row_data = []
    for row in table_body.find_all('tr'):
        col = row.find_all('td')
        col = [ele.text.strip() for ele in col]
        row_data.append(col)

    idx = pd.MultiIndex(levels=[fir,sec],codes=[[0,1,1,1,1,1,2,2,2,2,2,3,3,3,3,4,4,4,4],[0,5,1,2,3,4,5,1,2,3,4,5,6,7,8,5,6,7,8]], names=['lvl1','lvl2'])

    top_teams = pd.DataFrame(row_data[2:],columns=idx)

    top_teams['team'] = top_teams['team'].apply(lambda x: x.split('\n')[0])
    if date:
        top_teams['date'] = date

    #Flatten multiindex
    top_teams.columns = ['_'.join(x) for x in top_teams.columns.to_flat_index()]
    
    #Fix team column
    top_teams.rename({'team_':'team'},axis=1,inplace=True)
    
    return top_teams

In [3]:
df = get_top_elo()

In [4]:
def get_teams(pages=1):
    df = pd.DataFrame()
    for page in tqdm(range(pages)):
        request = requests.request('GET',f"https://api.opendota.com/api/teams?page={page}")
        data = request.json()
        df = pd.concat([df,pd.DataFrame(data)],ignore_index=True)
        sleep(1)
    df['team_id'] = df['team_id'].astype(int)
    return df

In [5]:
teams = get_teams(15)

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:35<00:00,  2.38s/it]


In [6]:
test = pd.merge(df,teams,how='left',left_on='team',right_on='name')

In [7]:
#test[test['name'].isna()]
test[['team','team_id']]

,team,team_id
0,PSG.LGD,15
1,PSG.LGD,7566630
2,PSG.LGD,6658355
3,PSG.LGD,8125994
4,OG,2586976
...,...,...
135,Simply TOOBASED,8272699
136,Arkosh Gaming,8180753
137,felt,8261882
138,5ManMidas,8230115


In [8]:
def get_team_players(team_id: int,current=None):
    request = requests.request('GET',f"https://api.opendota.com/api/teams/{team_id}/players")
    data = request.json()
    df = pd.DataFrame(data)
    if current:
        df = df[df['is_current_team_member']==True]
    
    return df

In [9]:
def get_all_team_players(team_id,current=None):
    all_players = pd.DataFrame()
    for team in tqdm(team_id,total=len(team_id)):
        player = get_team_players(team,current)
        player['team_id'] = team
        all_players = pd.concat([all_players, player],ignore_index=True)
        sleep(1)
    
    return all_players

In [10]:
players = get_all_team_players(list(test['team_id'].values),True)

100%|████████████████████████████████████████████████████████████████████████████████| 140/140 [04:30<00:00,  1.93s/it]


In [13]:
players.account_id.nunique()

508

In [14]:
players

,account_id,name,games_played,wins,is_current_team_member,team_id
0,898754153,萧瑟,472,328,True,15
1,111114687,y`,457,316,True,15
2,157475523,XinQ,457,316,True,15
3,118134220,Faith_bian,455,316,True,15
4,173978074,NothingToSay,416,289,True,15
...,...,...,...,...,...,...
503,175709801,DoubleKing,68,31,True,8686313
504,89354043,Mali burgos razbijač,50,27,True,8686313
505,79192770,Salacious Sea Lion 62,18,5,True,8686313
506,107746371,Atouf,18,5,True,8686313


In [15]:
players.to_csv('top_players.csv',index=False)

In [62]:
COLUMNS = ['Match']

def get_player_matches(account_ids):
    all_matches = pd.DataFrame(columns=COLUMNS)
    errors = []
    for account in tqdm(account_ids,total=len(account_ids)):
        try:
            response = requests.request('GET',f"https://www.datdota.com/players/{account}")
            soup = bs(response.content, "html.parser")
            table = soup.find_all('table')[0] # Find the first "table" tag in the page
            rows = table.find_all("tr")
            cy_data = []
            for row in rows:
                cells = row.find_all("td")[:1] # Get list of Most recent 100 Match Ids
                cy_data.append([cell.text for cell in cells]) # For each "td" tag, get the text inside it
            cy_data = pd.DataFrame(cy_data,columns=COLUMNS)
            cy_data = cy_data.loc[cy_data['Match'].notnull()].sort_values('Match',ascending=False).reset_index(drop=True)
            all_matches = pd.concat([all_matches,cy_data],ignore_index=True)
        except:
            errors.append(account)
    
    all_matches.drop_duplicates(inplace=True,ignore_index=True)
        
    return all_matches, errors

In [63]:
all_matches, errors = get_player_matches(players['account_id'].values) 

100%|████████████████████████████████████████████████████████████████████████████████| 495/495 [05:52<00:00,  1.40it/s]


In [61]:
len(all_matches)

10348

In [66]:
all_matches.to_csv('top_player_matches_7-24-2022.csv',index=False)

In [65]:
errors

[366081813, 254049656, 924306172]